# Question 1 — Interactive waste-bin map

**Aditi Saha · HS25M101**

This notebook makes a Folium map of the waste-bin points collected with QField.
It follows the same steps as Class Exercise 16, with a popup, photograph,
and collector name added for the assignment.


In [ ]:
# ------------------------------------------------------------
# 1. IMPORT LIBRARIES
# ------------------------------------------------------------
import os
import shutil

import geopandas as gpd
import folium


In [ ]:
# ------------------------------------------------------------
# 2. LOAD THE WASTE-BIN GEOPACKAGE
# ------------------------------------------------------------
data_path = r"C:\Users\Ā\Documents\python_class\HS25M101\HS25M101.gpkg"

dustbins = gpd.read_file(
    data_path,
    layer="notes_point"
)

# Photograph names are stored in a separate layer
attachments = gpd.read_file(
    data_path,
    layer="notes_attachments"
)


In [ ]:
# ------------------------------------------------------------
# 3. INSPECT CRS AND CONVERT TO EPSG:4326
# ------------------------------------------------------------
print(dustbins.crs)

dustbins = dustbins.to_crs(epsg=4326)

# Timestamp as text for the tooltip and popup
dustbins["timestamp"] = dustbins["timestamp"].astype(str)

# Fill empty titles and notes
dustbins["title"] = dustbins["title"].fillna("No title")
dustbins["note"] = dustbins["note"].fillna("No note")

# Student who collected the observations
dustbins["collector"] = "Aditi Saha"


In [ ]:
# ------------------------------------------------------------
# 4. JOIN THE PHOTOGRAPH NAMES TO EACH POINT
# ------------------------------------------------------------
# Keep one photograph name for each point uuid
photos = attachments[["note_uuid", "media"]].drop_duplicates("note_uuid")

dustbins = dustbins.merge(
    photos,
    left_on="uuid",
    right_on="note_uuid",
    how="left"
)

# QField stores paths like DCIM/JPEG_....jpg — keep only the file name
dustbins["photo_name"] = dustbins["media"].astype(str).str.split("/").str[-1]
dustbins.loc[dustbins["media"].isna(), "photo_name"] = ""

dustbins[["title", "note", "collector", "photo_name"]].head()


In [ ]:
# ------------------------------------------------------------
# 5. COPY PHOTOGRAPHS INTO THE WEB FOLDER
# ------------------------------------------------------------
# Popup uses photos/<file name>.
# After hosting, the browser loads these from the website, not from this PC.
os.makedirs("web/photos", exist_ok=True)

for name in dustbins["photo_name"]:
    if name == "":
        continue
    src = os.path.join("DCIM", name)
    dst = os.path.join("web", "photos", name)
    if os.path.exists(src):
        shutil.copy(src, dst)

print("Photographs copied to web/photos")


In [ ]:
# ------------------------------------------------------------
# 6. BUILD THE POPUP TEXT FOR EACH POINT
# ------------------------------------------------------------
popups = []

for i, row in dustbins.iterrows():
    html = (
        "<b>" + str(row["title"]) + "</b><br>"
        + "Collected by: " + str(row["collector"]) + "<br>"
        + "Note: " + str(row["note"]) + "<br>"
        + "Recorded at: " + str(row["timestamp"])
    )
    if row["photo_name"] != "":
        html = html + "<br><img src='photos/" + row["photo_name"] + "' width='220'>"
    else:
        html = html + "<br>No photograph attached"
    popups.append(html)

dustbins["popup"] = popups


In [ ]:
# ------------------------------------------------------------
# 7. CALCULATE MAP CENTRE
# ------------------------------------------------------------
center = [
    dustbins.geometry.y.mean(),
    dustbins.geometry.x.mean()
]
print(center)


In [ ]:
# ------------------------------------------------------------
# 8. CREATE THE FOLIUM MAP
# ------------------------------------------------------------
m = folium.Map(
    location=center,
    zoom_start=16,
    tiles=None
)

# Basemap 1 — OpenStreetMap
folium.TileLayer(
    tiles="OpenStreetMap",
    name="OpenStreetMap"
).add_to(m)

# Basemap 2 — Esri World Imagery
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri World Imagery",
    name="Satellite"
).add_to(m)


In [ ]:
# ------------------------------------------------------------
# 9. ADD WASTE-BIN POINTS WITH TOOLTIP AND POPUP
# ------------------------------------------------------------
# Same idea as Exercise 16: CircleMarker for each point.
# Tooltip = short info on hover
# Popup = full info + photograph on click

for i, row in dustbins.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=6,
        fill=True,
        fill_opacity=0.8,
        weight=1,
        color="#377eb8",
        fill_color="#377eb8",
        tooltip=str(row["title"]),
        popup=folium.Popup(row["popup"], max_width=280)
    ).add_to(m)


In [ ]:
# ------------------------------------------------------------
# 10. ADD LAYER CONTROL AND DISPLAY THE MAP
# ------------------------------------------------------------
folium.LayerControl().add_to(m)

m


In [ ]:
# ------------------------------------------------------------
# 11. SAVE THE MAP AS HTML FOR THE WEBSITE
# ------------------------------------------------------------
os.makedirs("web", exist_ok=True)
m.save("web/waste-bins.html")
print("Saved web/waste-bins.html")
